In [7]:
import pandas as pd

df = pd.read_csv(
    "../data/DSI_kickstarterscrape_dataset 2.csv",
    encoding="latin1"
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (45957, 17)


,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,138119,STRUM,http://www.kickstarter.com/projects/185476022/...,Film & Video,Animation,"Los Angeles, CA",live,20000.0,56.0,0.002800,3,"Fri, 08 Jun 2012 00:00:31 -0000",10,"$1,$10,$25,$40,$50,$100,$250,$1,000,$1,337,$9,001",1,0,28.00
3,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
4,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00


In [8]:
print(df.columns.tolist())

['project id', 'name', 'url', 'category', 'subcategory', 'location', 'status', 'goal', 'pledged', 'funded percentage', 'backers', 'funded date', 'levels', 'reward levels', 'updates', 'comments', 'duration']


In [9]:
categorical_cols = ["category", "subcategory", "location", "status"]

for col in categorical_cols:
    print(f"\n{col.upper()}")
    print("Unique values:", df[col].nunique())
    print("Missing values:", df[col].isna().sum())
    print(df[col].value_counts().head(10))


CATEGORY
Unique values: 14
Missing values: 0
category
Film &amp; Video    13053
Music               10913
Publishing           4770
Art                  3992
Theater              2492
Design               1768
Games                1738
Photography          1514
Food                 1439
Fashion              1136
Name: count, dtype: int64

SUBCATEGORY
Unique values: 51
Missing values: 0
subcategory
Documentary         4012
Short Film          3942
Music               3243
Film &amp; Video    2495
Theater             2492
Indie Rock          1939
Rock                1791
Narrative Film      1554
Photography         1514
Food                1439
Name: count, dtype: int64

LOCATION
Unique values: 4849
Missing values: 1322
location
Los Angeles, CA      3927
New York, NY         3647
Brooklyn, NY         1613
Chicago, IL          1496
San Francisco, CA    1350
Portland, OR          986
Seattle, WA           949
Austin, TX            825
Boston, MA            784
Nashville, TN         680
Na

In [10]:
print("Dataset shape:", df.shape)
print("\nStatus distribution:")
print(df["status"].value_counts())

Dataset shape: (45957, 17)

Status distribution:
status
successful    22969
failed        18996
live           3929
canceled         59
suspended         4
Name: count, dtype: int64


## Prepare Location Features

The original dataset stores city and state together in the `location` column. 
We separate these into individual city and state features so they can be
analyzed and encoded independently.

In [12]:
df[["city", "state"]] = df["location"].str.rsplit(",", n=1, expand=True)

df["city"] = df["city"].str.strip()
df["state"] = df["state"].str.strip()

df[["location", "city", "state"]].head(10)

,location,city,state
0,"Columbia, MO",Columbia,MO
1,"Maplewood, NJ",Maplewood,NJ
2,"Los Angeles, CA",Los Angeles,CA
3,"Los Angeles, CA",Los Angeles,CA
4,"Novi, MI",Novi,MI
5,"Portland, OR",Portland,OR
6,"Collegedale, TN",Collegedale,TN
7,"Chicago, IL",Chicago,IL
8,"Chicago, IL",Chicago,IL
9,"Chicago, IL",Chicago,IL


In [13]:
print("Unique cities:", df["city"].nunique())
print("Unique states:", df["state"].nunique())

print("\nMissing cities:", df["city"].isna().sum())
print("Missing states:", df["state"].isna().sum())

print("\nTop states:")
print(df["state"].value_counts().head(15))

Unique cities: 4075
Unique states: 198

Missing cities: 1322
Missing states: 1323

Top states:
state
CA    8973
NY    7084
TX    2056
IL    1887
FL    1588
WA    1426
PA    1328
MA    1314
OR    1300
GA    1058
MI    1023
TN    1017
NC     842
OH     841
CO     807
Name: count, dtype: int64


## Clean Categorical Variables

Before encoding, categorical features are standardized by removing extra whitespace
and decoding HTML entities. The original categorical columns are retained for EDA
and interpretation.

In [14]:
import html

cols_to_clean = ["category", "subcategory", "city", "state"]

for col in cols_to_clean:
    df[col] = df[col].apply(
        lambda x: html.unescape(x.strip()) if isinstance(x, str) else x
    )

df[cols_to_clean].head()

,category,subcategory,city,state
0,Film & Video,Short Film,Columbia,MO
1,Games,Board & Card Games,Maplewood,NJ
2,Film & Video,Animation,Los Angeles,CA
3,Film & Video,Documentary,Los Angeles,CA
4,Fashion,Fashion,Novi,MI


In [15]:
for col in cols_to_clean:
    print(f"{col}: {df[col].nunique()} unique values")

print("\nCategories:")
print(df["category"].value_counts())

category: 13 unique values
subcategory: 49 unique values
city: 4075 unique values
state: 198 unique values

Categories:
category
Film & Video    13551
Music           10913
Publishing       4770
Art              3992
Theater          2492
Design           1768
Games            1738
Photography      1514
Food             1439
Fashion          1136
Comics           1072
Technology        811
Dance             761
Name: count, dtype: int64


## Categorical Variable Preparation

The categorical features used for preprocessing are category, subcategory,
city, and state. Location is separated into city and state before encoding.

Low-cardinality categorical features can be one-hot encoded directly, while
city requires additional consideration because of its high cardinality.

In [16]:
categorical_features = ["category", "subcategory", "city", "state"]

for col in categorical_features:
    print(f"{col}:")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Missing values: {df[col].isna().sum()}")

category:
  Unique values: 13
  Missing values: 0
subcategory:
  Unique values: 49
  Missing values: 0
city:
  Unique values: 4075
  Missing values: 1322
state:
  Unique values: 198
  Missing values: 1323


In [17]:
city_counts = df["city"].value_counts()

print("Total unique cities:", df["city"].nunique())
print("Cities appearing only once:", (city_counts == 1).sum())
print("Cities with fewer than 10 projects:", (city_counts < 10).sum())

print("\nTop 20 cities:")
print(city_counts.head(20))

Total unique cities: 4075
Cities appearing only once: 2019
Cities with fewer than 10 projects: 3655

Top 20 cities:
city
Los Angeles      3927
New York         3650
Brooklyn         1614
Chicago          1496
San Francisco    1350
Portland         1103
Seattle           949
Austin            826
Boston            785
Nashville         682
Atlanta           612
Philadelphia      597
Minneapolis       505
Washington        477
San Diego         418
Denver            391
Detroit           352
Orlando           333
Dallas            328
New Orleans       327
Name: count, dtype: int64


## Handle High-Cardinality City Values

City contains thousands of unique values, many of which occur only a few times.
To reduce dimensionality before encoding, cities with fewer than 10 campaigns
are grouped into an `Other` category.

In [18]:
city_counts = df["city"].value_counts()
frequent_cities = city_counts[city_counts >= 10].index

df["city_grouped"] = df["city"]

df.loc[
    df["city"].notna() & ~df["city"].isin(frequent_cities),
    "city_grouped"
] = "Other"

print("Original unique cities:", df["city"].nunique())
print("Grouped unique cities:", df["city_grouped"].nunique())
print("Missing grouped cities:", df["city_grouped"].isna().sum())

df["city_grouped"].value_counts().head(15)

Original unique cities: 4075
Grouped unique cities: 421
Missing grouped cities: 1322


city_grouped
Other            7783
Los Angeles      3927
New York         3650
Brooklyn         1614
Chicago          1496
San Francisco    1350
Portland         1103
Seattle           949
Austin            826
Boston            785
Nashville         682
Atlanta           612
Philadelphia      597
Minneapolis       505
Washington        477
Name: count, dtype: int64

## One-Hot Encode Categorical Features

The prepared categorical variables are converted into numeric features using
one-hot encoding. Unknown categories are ignored so the preprocessing step can
handle new values during future model evaluation.

In [22]:
from sklearn.preprocessing import OneHotEncoder

features_to_encode = [
    "category",
    "subcategory",
    "city_grouped",
    "state"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_data = encoder.fit_transform(df[features_to_encode])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(features_to_encode),
    index=df.index
)

print("Original categorical shape:", df[features_to_encode].shape)
print("Encoded shape:", encoded_df.shape)

encoded_df.head()

Original categorical shape: (45957, 4)
Encoded shape: (45957, 683)


,category_Art,category_Comics,category_Dance,category_Design,category_Fashion,category_Film & Video,category_Food,category_Games,category_Music,category_Photography,...,state_Venezuela,state_Viet Nam,state_Virginia,state_WA,state_WI,state_WV,state_WY,state_Zambia,state_Zimbabwe,state_nan
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
print("Number of encoded features:", encoded_df.shape[1])
print("Missing values in encoded data:", encoded_df.isna().sum().sum())

print("\nEncoded feature counts:")
for feature in features_to_encode:
    count = sum(col.startswith(feature + "_") for col in encoded_df.columns)
    print(f"{feature}: {count}")

Number of encoded features: 683
Missing values in encoded data: 0

Encoded feature counts:
category: 13
subcategory: 49
city_grouped: 422
state: 199


## Summary

Categorical variables were prepared for downstream EDA and machine learning.

- `location` was separated into `city` and `state`.
- HTML entities and extra whitespace were cleaned from categorical values.
- `category` and `subcategory` were standardized before encoding.
- Rare cities with fewer than 10 campaigns were grouped into `Other` to reduce high cardinality.
- `category`, `subcategory`, `city_grouped`, and `state` were one-hot encoded using `OneHotEncoder` with unknown-category handling.
- The preprocessing logic is designed to be rerun on the finalized dataset produced by the upstream data preparation tasks.